## MONAI Integration
MONAI allows the definition of AI models using the "bundle" concept. It allows for easy experimentation and sharing of models that have been developed using MONAI. Using the bundle configurations, we can use MONAI's MonaiAlgo (the implementation of ClientAlgo) to execute a bundle model in a federated scenario using NVFlare.

You can find the code for the [MONAI Integration code](https://github.com/NVIDIA/NVFlare/blob/dev/integration/monai/README.md) in the `integration` directory of the NVFlare GitHub.

There is also an example that walks through preparing the environment and input datasets, and running the [MONAI Spleen CT Segmentation example](https://github.com/NVIDIA/NVFlare/tree/dev/integration/monai/examples/spleen_ct_segmentation) using a locally provisioned secure deployment.  We'll walk through that here.

## Preparing the POC environment
Before running the POC, there are a couple important environment variables that should be set.

First, to simplify deploying the example apps in the NVFlare GitHub repo, you can set `NVFLARE_HOME` to the root of the GitHub clone.  In this case, we've cloned to our current working directory, so we can set it as:

In [1]:
import os
workdir=os.getcwd()
%env NVFLARE_HOME={workdir}/../NVFlare

env: NVFLARE_HOME=/flare/notebooks/../NVFlare


By default, POC mode uses a temporary workspace in /tmp/nvflare/poc.  We would like to keep the workspace within our working directory, so let's create a poc_workspace dir.  We can then use the `NVFLARE_POC_WORKSPACE` variable to define this as the POC workspace.

Note:  if you have previously created the poc_workspace, you will want to clean it up using the `nvflare poc --clean` command.

In [3]:
# !nvflare poc --clean
!mkdir monai_poc_workspace
%env NVFLARE_POC_WORKSPACE={workdir}/monai_poc_workspace

env: NVFLARE_POC_WORKSPACE=/flare/notebooks/monai_poc_workspace


### Preparing the POC workspace

Now that we've configured out POC environment, we can prepare the POC workspace.  By default, this will generate POC packages for a server and two clients.

(Note that `nvflare poc --prepare` prompts you to create the workspace.)

In [4]:
!printf '%s\n' y | nvflare poc --prepare

prepare_poc at /flare/notebooks/monai_poc_workspace for 2 clients
This will delete poc folder in /flare/notebooks/monai_poc_workspace directory and create a new one. Is it OK to proceed? (y/N) Successfully creating poc folder at /flare/notebooks/monai_poc_workspace.  Please read poc/Readme.rst for user guide.


******* Files generated by this poc command are NOT intended for production environments.
link examples from /flare/notebooks/../NVFlare/examples to /flare/notebooks/monai_poc_workspace/admin/transfer


Let's take a look.

In [5]:
!tree monai_poc_workspace

monai_poc_workspace
├── Readme.rst
├── admin
│   ├── local
│   ├── startup
│   │   ├── fed_admin.json
│   │   └── fl_admin.sh
│   └── transfer -> /flare/notebooks/../NVFlare/examples
├── server
│   ├── local
│   │   ├── log.config
│   │   └── resources.json
│   └── startup
│       ├── fed_server.json
│       ├── start.sh
│       ├── stop_fl.sh
│       └── sub_start.sh
├── site-1
│   ├── local
│   │   ├── log.config
│   │   └── resources.json
│   └── startup
│       ├── fed_client.json
│       ├── start.sh
│       ├── stop_fl.sh
│       └── sub_start.sh
└── site-2
    ├── local
    │   ├── log.config
    │   └── resources.json
    └── startup
        ├── fed_client.json
        ├── start.sh
        ├── stop_fl.sh
        └── sub_start.sh

13 directories, 21 files


### Running the POC Deployment

When starting the POC deployment, it's necessary to use a separate terminal since the `nvflare poc --start` command will run  in the foreground emitting output from the server and any connected clients.

Also note that `nvflare poc --start` starts all participants, including the admin console.  It's often nice to start server and clients separately so that we can interact with the deployment using a separate admin console.  To do this, we'll pass the `-ex admin` arg to exclude the admin client from the initial POC run and use the FLARE API to run admin commands separately.

So pop open the launcher, launch a terminal, and run (remembering to set the NVFLARE_POC_WORKSPACE and NVFLARE_HOME vars!):

```shell
export NVFLARE_POC_WORKSPACE=$(pwd -P)/notebooks/monai_poc_workspace
export NVFLARE_HOME=$(pwd -P)/NVFlare
nvflare poc --start -ex admin
```

Keep this terminal open so you can continue to watch server and client output.

## Setting up the MONAI example

The MONAI example for a distributed (or local Docker Compose "distriburted" deployment) can be found in the `NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world` directory.    Let's copy it to our `notebooks/examples` directory.

In [8]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/spleen_ct_segmentation_real-world ]; then \
    cp -r ../NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world examples/; fi
!tree examples/spleen_ct_segmentation_real-world

examples/spleen_ct_segmentation_real-world
├── README.md
├── download_spleen_dataset.py
├── job
│   ├── app
│   │   └── config
│   │       ├── config_fed_client.json
│   │       └── config_fed_server.json
│   └── meta.json
├── job_he
│   ├── app
│   │   └── config
│   │       ├── config_fed_client.json
│   │       └── config_fed_server.json
│   └── meta.json
├── start_fl_secure.sh
├── submit_job.py
├── submit_job.sh
├── virtualenv
│   ├── requirements.txt
│   └── set_env.sh
└── workspaces
    └── secure_project.yml

8 directories, 14 files


We now need to download the MONAI bundle that contains the `spleen_ct_segmentation` model and configuration.  We can do this using the MONAI bundle download built-in script.

This model and configuration will be part of the FLARE spleen_ct_segmentation_real-world app that is deployed to the FLARE clients, so we'll provde the job directory as the bundle download path.

We will also download the example spleen dataset and push it to a directory accessible by the containers running the FLARE clients.

In [12]:
!pwd

/flare/notebooks


In [16]:
%env JOB_DIR=examples/spleen_ct_segmentation_real-world/job
!python3 -m monai.bundle download \
    --name "spleen_ct_segmentation" \
    --version "0.3.7" \
    --bundle_dir ./${JOB_DIR}/app/config
!if [ ! -d data/Task09_Spleen ]; then \
    python3 examples/spleen_ct_segmentation_real-world/download_spleen_dataset.py; fi
#!for site in site-1 site-2; do mkdir monai_poc_workspace/${site}/data; cp -r data/Task09_Spleen monai_poc_workspace/${site}/data; done
!sed -i "s,/workspace/data/Task09_Spleen,${PWD}/data/Task09_Spleen,g" ${JOB_DIR}/app/config/spleen_ct_segmentation/configs/train.json

env: JOB_DIR=examples/spleen_ct_segmentation_real-world/job
2023-03-13 14:19:26,122 - INFO - --- input summary of monai.bundle.scripts.download ---
2023-03-13 14:19:26,122 - INFO - > name: 'spleen_ct_segmentation'
2023-03-13 14:19:26,122 - INFO - > version: '0.3.7'
2023-03-13 14:19:26,122 - INFO - > bundle_dir: './examples/spleen_ct_segmentation_real-world/job/app/config'
2023-03-13 14:19:26,123 - INFO - > source: 'github'
2023-03-13 14:19:26,123 - INFO - > repo: 'Project-MONAI/model-zoo/hosting_storage_v1'
2023-03-13 14:19:26,123 - INFO - > progress: True
2023-03-13 14:19:26,123 - INFO - ---


2023-03-13 14:19:26,123 - INFO - Expected md5 is None, skip md5 check for file examples/spleen_ct_segmentation_real-world/job/app/config/spleen_ct_segmentation_v0.3.7.zip.
2023-03-13 14:19:26,123 - INFO - File exists: examples/spleen_ct_segmentation_real-world/job/app/config/spleen_ct_segmentation_v0.3.7.zip, skipped downloading.
2023-03-13 14:19:26,123 - INFO - Writing into directory: examples/

## Running the MONAI app
Now that we have the example spleen segmentation app in `examples/spleen_ct_segmentation_real-world` and the MONAI bundle config in the `job/app/config` subdirectory, we can look at how we configure the job parameters with FLARE MONAI FL integration.

In [22]:
!tree examples/spleen_ct_segmentation_real-world

examples/spleen_ct_segmentation_real-world
├── README.md
├── download_spleen_dataset.py
├── job
│   ├── app
│   │   └── config
│   │       ├── config_fed_client.json
│   │       ├── config_fed_server.json
│   │       ├── spleen_ct_segmentation
│   │       │   ├── LICENSE
│   │       │   ├── configs
│   │       │   │   ├── evaluate.json
│   │       │   │   ├── inference.json
│   │       │   │   ├── logging.conf
│   │       │   │   ├── metadata.json
│   │       │   │   ├── multi_gpu_evaluate.json
│   │       │   │   ├── multi_gpu_train.json
│   │       │   │   └── train.json
│   │       │   ├── docs
│   │       │   │   ├── README.md
│   │       │   │   └── data_license.txt
│   │       │   └── models
│   │       │       ├── model.pt
│   │       │       └── model.ts
│   │       └── spleen_ct_segmentation_v0.3.7.zip
│   └── meta.json
├── job_he
│   ├── app
│   │   └── config
│   │       ├── config_fed_client.json
│   │       └── config_fed_server.json
│   └── meta.json
├── start_fl_secure.s

As with other FLARE applications, we have the usual `config_fed_client.json` and `config_fed_server.json` configurations.  We also have a set of MONAI configs from the MONAI Bundle in `job/app/config/spleen_ct_segmentation/configs`.

The `job/app/config/spleen_ct_segmentation/configs/train.json` defines the MONAI training parameters, for example inputs/output and network definition (you can navigate in the file browser and open the train.json file in the Editor to see the full configuration:

```json
    "bundle_root": "/workspace/data/spleen_ct_segmentation",
    "ckpt_dir": "$@bundle_root + '/models'",
    "output_dir": "$@bundle_root + '/eval'",
    "dataset_dir": "/workspace/data/Task09_Spleen",
    "images": "$list(sorted(glob.glob(@dataset_dir + '/imagesTr/*.nii.gz')))",
    "labels": "$list(sorted(glob.glob(@dataset_dir + '/labelsTr/*.nii.gz')))",
    "val_interval": 5,
    "device": "$torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')",
    "network_def": {
        "_target_": "UNet",
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 2,
        "channels": [
            16,
            32,
            64,
            128,
            256
        ],
        "strides": [
            2,
            2,
            2,
            2
        ],
        "num_res_units": 2,
        "norm": "batch"
    },
```

The MONAI Client executor uses this config along with the parameters in `config_fed_client.json` and `config_fed_server.json` to define the FLARE app.  For example, the client config defines this executor and the number of local training epochs:
```json
{
  "format_version": 2,

  "executors": [
    {
      "tasks": [
        "train", "submit_model", "validate"
      ],
      "executor": {
        "id": "executor",
        "path": "monai_nvflare.client_algo_executor.ClientAlgoExecutor",
        "args": {
          "client_algo_id": "client_algo",
          "key_metric": "val_mean_dice"
        }
      }
    }
  ],

  "task_result_filters": [
  ],
  "task_data_filters": [
  ],

  "components": [
    {
      "id": "client_algo",
      "path": "monai.fl.client.MonaiAlgo",
      "args": {
        "bundle_root": "config/spleen_ct_segmentation",
        "local_epochs": 10
      }
    }
  ]
}
```

Similarly, the server config defines the number of global rounds:
```json
{
  "format_version": 2,

  "min_clients": 2,
  "num_rounds": 100,

  "task_data_filters": [],
  "task_result_filters": [],
  "components": [

  ... <snip> ...
  
  ],
  "workflows": [
      {
        "id": "scatter_gather_ctl",
        "name": "ScatterAndGather",
        "args": {
            "min_clients" : "{min_clients}",
            "num_rounds" : "{num_rounds}",
            "start_round": 0,
            "wait_time_after_min_received": 10,
            "aggregator_id": "aggregator",
            "persistor_id": "persistor",
            "shareable_generator_id": "shareable_generator",
            "train_task_name": "train",
            "train_timeout": 0
        }
  ...
```

### Exercise
The server is configured to for 100 rounds of training, which will take a long time to complete.  Use the file browser to the left to navigate to `examples/spleen_ct_segmentation_real-world/job/app/config/` and right-click to open `config_fed_server.json` in the Editor.  Change `"num_rounds": 100` to something much smaller, like `"num_rounds": 2`.


### Using the FLARE API to connect and submit the MONAI app

The admin directory contains the startup script for the FLARE Console, which can be used interactively to operate a running FLARE deployment.  A FLARE deployment can also be managed using the FLARE API, which will use the configuration in the admin directory to connect to the FLARE server.  Since we already have the server and clients running in the background from the above terminal commands, we'll use FLARE API to start a new admin session and connect.

To get started, we need to import the FLARE API class and initialize session.

In [17]:
from nvflare.fuel.flare_api.flare_api import new_insecure_session

admin_session = new_insecure_session(startup_kit_location = workdir + "/monai_poc_workspace/admin")
print(admin_session.get_system_info())

SystemInfo
server_info:
status: stopped, start_time: Mon Mar 13 13:58:51 2023
client_info:
site-1(last_connect_time: Mon Mar 13 14:19:48 2023)
site-2(last_connect_time: Mon Mar 13 14:19:41 2023)
job_info:




Now with the session connected, we can use the job we've just configured to submit and run the MONAI application.

In [18]:
path_to_job_config = "/flare/notebooks/examples/spleen_ct_segmentation_real-world/job"
job_id = admin_session.submit_job(path_to_job_config)
print(job_id)

09d0fca2-f911-4130-bc13-5b9b8fbb47df



After the job has been submitted, you can navigate to the terminal that was used to start the POC deployment and see progress messages as the job executes.

You can also use the admin session to query job status as in the previous POC notebook.

In [20]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)

Job Status
[
  {
    "job_id": "09d0fca2-f911-4130-bc13-5b9b8fbb47df",
    "job_name": "spleen-bundle",
    "status": "RUNNING",
    "submit_time": "2023-03-13T14:20:01.991389+00:00",
    "duration": "0:42:46.353518"
  },
  {
    "job_id": "b55d98e1-a91c-483d-812f-3aa4431906d4",
    "job_name": "spleen-bundle",
    "status": "FINISHED:EXECUTION_EXCEPTION",
    "submit_time": "2023-03-13T14:06:02.376413+00:00",
    "duration": "0:00:15.963064"
  },
  {
    "job_id": "3739db81-35bc-4eae-9a9e-11581aee7e81",
    "job_name": "hello-pt",
    "status": "FINISHED:COMPLETED",
    "submit_time": "2023-03-10T02:26:13.702133+00:00",
    "duration": "0:03:49.094394"
  },
  {
    "job_id": "86b5e7e3-69b1-4d2c-a0f0-643e9fbca643",
    "job_name": "hello-pt",
    "status": "FINISHED:COMPLETED",
    "submit_time": "2023-03-09T17:08:59.671175+00:00",
    "duration": "0:03:13.037359"
  },
  {
    "job_id": "426cfd61-409f-438b-8efd-5eb6517e3f65",
    "job_name": "hello-pt",
    "status": "FINISHED:COMPLETE

{'name': 'spleen-bundle',
 'resource_spec': {},
 'min_clients': 2,
 'deploy_map': {'app': ['@ALL']},
 'job_folder_name': 'job',
 'submitter_name': 'admin',
 'submitter_org': 'global',
 'submitter_role': 'super',
 'job_id': '09d0fca2-f911-4130-bc13-5b9b8fbb47df',
 'submit_time': 1678717201.9913893,
 'submit_time_iso': '2023-03-13T14:20:01.991389+00:00',
 'start_time': '2023-03-13 14:20:04.509291',
 'duration': 'N/A',
 'status': 'RUNNING',
 'job_deploy_detail': ['server: OK', 'site-1: OK', 'site-2: OK'],
 'schedule_count': 1,
 'last_schedule_time': 1678717202.8857994,
 'schedule_history': ['2023-03-13 14:20:02: scheduled']}

In [21]:
admin_session.abort_job(job_id)